# Incremental Evaluation Benchmarks

Measures per-batch latency as new facts arrive incrementally.

All benchmarks use the Free Join backend.

Three evaluation modes:
- **Semi-naive (incremental)**: `poll()` processes only new deltas, full materialization
- **MST (incremental)**: Transformed program, persistent runtime, delta-driven, query-directed
- **SDT (incremental)**: SDT-transformed program, persistent runtime, delta-driven, query-directed

Plus single-shot MST/SDT for comparison (creates fresh runtime each batch).

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd

from pymycrodatalog import IncrementalQueryView, MicroRuntime, Variable

X = Variable("X")
Y = Variable("Y")
Z = Variable("Z")

LINEAR_TC = [
    (("tc", (X, Y)), ("e", (X, Y))),
    (("tc", (X, Z)), ("e", (X, Y)), ("tc", (Y, Z))),
]

ANCESTOR = [
    (("ancestor", (X, Y)), ("parent", (X, Y))),
    (("ancestor", (X, Z)), ("parent", (X, Y)), ("ancestor", (Y, Z))),
]

## Helpers

In [ ]:
def generate_batches(
    total_edges: int, batch_size: int, n_nodes: int
) -> list[list[tuple[str, tuple[int, int]]]]:
    edges = []
    seen: set[tuple[int, int]] = set()
    for i in range(total_edges * 3):
        s, d = (i * 13 + 7) % n_nodes, (i * 31 + 11) % n_nodes
        if s != d and (s, d) not in seen:
            seen.add((s, d))
            edges.append(("e", (s, d)))
        if len(edges) >= total_edges:
            break
    return [edges[i : i + batch_size] for i in range(0, len(edges), batch_size)]


def bench_semi_naive_incremental(
    rules: list[tuple],
    batches: list[list[tuple]],
    pred: str,
    pattern: tuple,
) -> list[dict]:
    """Semi-naive poll() — full materialization, incremental deltas."""
    rt = MicroRuntime(rules, engine="free_join")
    rows = []
    cumulative = 0
    for batch_idx, batch in enumerate(batches):
        for f in batch:
            rt.insert(f)
        cumulative += len(batch)
        t0 = time.perf_counter_ns()
        rt.poll()
        results = rt.query(pred, pattern)
        elapsed_us = (time.perf_counter_ns() - t0) / 1000
        rows.append({"batch": batch_idx, "cumulative_facts": cumulative, "result_count": len(results), "time_us": elapsed_us})
    return rows


def bench_incremental_view(
    rules: list[tuple],
    batches: list[list[tuple]],
    pred: str,
    pattern: tuple,
    strategy: str = "MST",
) -> list[dict]:
    """IncrementalQueryView — persistent transformed runtime, delta-driven."""
    view = IncrementalQueryView(rules, pred, pattern, strategy=strategy, engine="free_join")
    rows = []
    cumulative = 0
    for batch_idx, batch in enumerate(batches):
        for f in batch:
            view.insert(f)
        cumulative += len(batch)
        t0 = time.perf_counter_ns()
        view.poll()
        results = view.query()
        elapsed_us = (time.perf_counter_ns() - t0) / 1000
        rows.append({"batch": batch_idx, "cumulative_facts": cumulative, "result_count": len(results), "time_us": elapsed_us})
    return rows


def bench_single_shot(
    rules: list[tuple],
    batches: list[list[tuple]],
    pred: str,
    pattern: tuple,
    strategy: str = "Bottom-up",
) -> list[dict]:
    """Non-incremental query_program — fresh runtime each batch."""
    rt = MicroRuntime(rules, engine="free_join")
    rows = []
    cumulative = 0
    for batch_idx, batch in enumerate(batches):
        for f in batch:
            rt.insert(f)
        cumulative += len(batch)
        t0 = time.perf_counter_ns()
        results = rt.query_program(pred, pattern, rules, strategy)
        elapsed_us = (time.perf_counter_ns() - t0) / 1000
        rows.append({"batch": batch_idx, "cumulative_facts": cumulative, "result_count": len(results), "time_us": elapsed_us})
    return rows

## Benchmark 1: TC BF \u2014 streaming edges, 50 nodes, batch=10

In [ ]:
batches_50 = generate_batches(200, 10, 50)
print(f"{len(batches_50)} batches, {sum(len(b) for b in batches_50)} total edges")

r1 = {}
r1["Semi-naive (incr)"] = bench_semi_naive_incremental(LINEAR_TC, batches_50, "tc", (0, None))
r1["MST (incr)"] = bench_incremental_view(LINEAR_TC, batches_50, "tc", (0, None), "MST")
r1["SDT (incr)"] = bench_incremental_view(LINEAR_TC, batches_50, "tc", (0, None), "SDT")
r1["MST (fresh)"] = bench_single_shot(LINEAR_TC, batches_50, "tc", (0, None), "Bottom-up")
r1["SDT (fresh)"] = bench_single_shot(LINEAR_TC, batches_50, "tc", (0, None), "SDT")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for name, rows in r1.items():
    df = pd.DataFrame(rows)
    ax1.plot(df["cumulative_facts"], df["time_us"], marker=".", label=name, markersize=3)
    ax2.plot(df["cumulative_facts"], df["result_count"], marker=".", label=name, markersize=3)
ax1.set_xlabel("Cumulative edges")
ax1.set_ylabel("Per-batch time (\u00b5s)")
ax1.set_title("Per-batch latency")
ax1.legend(fontsize=8)
ax1.set_yscale("log")
ax2.set_xlabel("Cumulative edges")
ax2.set_ylabel("Result count")
ax2.set_title("Results over time")
ax2.legend(fontsize=8)
fig.suptitle("Incremental TC(0, _) \u2014 50 nodes, batch=10", fontsize=13)
fig.tight_layout()
plt.show()

## Benchmark 2: Larger graph \u2014 100 nodes, 500 edges, batch=20

In [ ]:
batches_100 = generate_batches(500, 20, 100)
print(f"{len(batches_100)} batches, {sum(len(b) for b in batches_100)} total edges")

r2 = {}
r2["Semi-naive (incr)"] = bench_semi_naive_incremental(LINEAR_TC, batches_100, "tc", (0, None))
r2["MST (incr)"] = bench_incremental_view(LINEAR_TC, batches_100, "tc", (0, None), "MST")
r2["SDT (incr)"] = bench_incremental_view(LINEAR_TC, batches_100, "tc", (0, None), "SDT")
r2["MST (fresh)"] = bench_single_shot(LINEAR_TC, batches_100, "tc", (0, None), "Bottom-up")

fig, ax = plt.subplots(figsize=(10, 5))
for name, rows in r2.items():
    df = pd.DataFrame(rows)
    ax.plot(df["cumulative_facts"], df["time_us"], marker=".", label=name, markersize=3)
ax.set_xlabel("Cumulative edges")
ax.set_ylabel("Per-batch time (\u00b5s)")
ax.set_title("Incremental TC(0, _) \u2014 100 nodes, 500 edges, batch=20")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## Benchmark 3: Incremental vs Fresh MST \u2014 speedup over time

In [ ]:
incr_df = pd.DataFrame(r1["MST (incr)"])
fresh_df = pd.DataFrame(r1["MST (fresh)"])
speedup = fresh_df["time_us"].values / incr_df["time_us"].values

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(incr_df["cumulative_facts"], speedup, marker="o", markersize=4, color="green")
ax.axhline(y=1, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Cumulative edges")
ax.set_ylabel("Speedup (fresh / incremental)")
ax.set_title("MST Incremental Speedup Over Fresh")
plt.tight_layout()
plt.show()

print(f"Average speedup: {speedup.mean():.2f}x")
print(f"Max speedup: {speedup.max():.2f}x (at batch {speedup.argmax()})")

## Benchmark 4: Incremental Ancestor BB \u2014 growing family chain

In [ ]:
n = 30
parent_batches = [[("parent", (i, i + 1))] for i in range(n - 1)]

r3 = {}
r3["Semi-naive (incr)"] = bench_semi_naive_incremental(ANCESTOR, parent_batches, "ancestor", (0, n - 1))
r3["MST (incr)"] = bench_incremental_view(ANCESTOR, parent_batches, "ancestor", (0, n - 1), "MST")
r3["MST (fresh)"] = bench_single_shot(ANCESTOR, parent_batches, "ancestor", (0, n - 1), "Bottom-up")

fig, ax = plt.subplots(figsize=(10, 5))
for name, rows in r3.items():
    df = pd.DataFrame(rows)
    ax.plot(df["cumulative_facts"], df["time_us"], marker=".", label=name, markersize=4)
ax.set_xlabel("Chain length")
ax.set_ylabel("Per-batch time (\u00b5s)")
ax.set_title(f"Incremental Ancestor BB: ancestor(0, {n - 1})")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

In [ ]:
summary_rows = []
for name, rows in r1.items():
    total = sum(r["time_us"] for r in rows)
    final = rows[-1]["result_count"]
    summary_rows.append({"strategy": name, "total_us": f"{total:.0f}", "final_results": final})

print("TC BF 50-node graph, 200 edges, 20 batches:")
print(pd.DataFrame(summary_rows).to_string(index=False))